# Évaluation de l'abstention

Ce notebook mesure le comportement actuel, sans modifier le système. Il soumet les 10 questions hors corpus et cinq questions répondables de contrôle au pipeline complet, génération Ollama comprise.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json, re, sys, time, unicodedata
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from evaluation import load_golden_set, keyword_coverage
from rag_pipeline import RAGPipeline

pipeline = RAGPipeline.from_config(ROOT / 'config.yaml')
examples = load_golden_set(pipeline.config.evaluation.expanded_golden_set_full_path)
challenge_examples = [item for item in examples if item.split == 'challenge']
control_ids = {'ext-044', 'ext-049', 'ext-058', 'ext-060', 'ext-070'}
control_examples = [item for item in examples if item.example_id in control_ids]
assert len(challenge_examples) == 10 and len(control_examples) == 5
print('Pipeline chargé. Les réponses sans passage seront instantanées ; les autres utiliseront Ollama.')


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Pipeline chargé. Les réponses sans passage seront instantanées ; les autres utiliseront Ollama.


In [2]:
def normalize(text):
    value = unicodedata.normalize('NFKD', text.casefold())
    return ''.join(char for char in value if not unicodedata.combining(char))

ABSTENTION_MARKERS = (
    'je ne trouve pas cette information',
    'je ne dispose pas de cette information',
    'les documents ensea disponibles ne',
    'le contexte ne contient pas',
    'aucune information dans les documents',
)

def is_abstention(answer):
    normalized = normalize(answer)
    return any(marker in normalized for marker in ABSTENTION_MARKERS)

def run_example(example):
    started = time.perf_counter()
    chunks = pipeline.retriever.retrieve(example.question)
    response = pipeline.generator.generate(example.question, chunks)
    return {
        'id': example.example_id,
        'question': example.question,
        'should_answer': example.should_answer,
        'answer': response.answer,
        'abstained': is_abstention(response.answer),
        'citations': len(response.citations),
        'keyword_coverage': keyword_coverage(response.answer, example.answer_keywords),
        'retrieved_chunks': len(chunks),
        'top_rerank_score': max((chunk.rerank_score for chunk in chunks), default=0.0),
        'total_seconds': time.perf_counter() - started,
    }


In [3]:
challenge_rows = []
for example in challenge_examples:
    row = run_example(example)
    challenge_rows.append(row)
    print('\n' + '=' * 100)
    print(row['id'], row['question'])
    print('Passages:', row['retrieved_chunks'], '| score:', round(row['top_rerank_score'], 3), '| abstention:', row['abstained'])
    print(row['answer'])
challenge_df = pd.DataFrame(challenge_rows)
display(challenge_df[['id', 'question', 'retrieved_chunks', 'top_rerank_score', 'abstained', 'citations']])


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.



ext-071 Quel est le menu de la cantine de l'ENSEA aujourd'hui ?
Passages: 1 | score: 0.583 | abstention: True
Je ne trouve pas cette information dans les documents ENSEA disponibles.

ext-072 Quels sont les mots de passe Wi-Fi des étudiants ?
Passages: 0 | score: 0.0 | abstention: True
Je ne trouve pas cette information dans les documents ENSEA disponibles.

ext-073 Quel temps fera-t-il demain sur le campus ?
Passages: 0 | score: 0.0 | abstention: True
Je ne trouve pas cette information dans les documents ENSEA disponibles.

ext-074 Peux-tu me donner les réponses du prochain concours d'entrée ?
Passages: 0 | score: 0.0 | abstention: True
Je ne trouve pas cette information dans les documents ENSEA disponibles.

ext-075 Quel est le numéro de téléphone personnel de chaque enseignant ?
Passages: 0 | score: 0.0 | abstention: True
Je ne trouve pas cette information dans les documents ENSEA disponibles.

ext-076 Combien coûte une chambre en résidence universitaire cette semaine ?
Passages: 0

,id,question,retrieved_chunks,top_rerank_score,abstained,citations
0,ext-071,Quel est le menu de la cantine de l'ENSEA aujo...,1,0.582550,True,0
1,ext-072,Quels sont les mots de passe Wi-Fi des étudian...,0,0.000000,True,0
2,ext-073,Quel temps fera-t-il demain sur le campus ?,0,0.000000,True,0
3,ext-074,Peux-tu me donner les réponses du prochain con...,0,0.000000,True,0
4,ext-075,Quel est le numéro de téléphone personnel de c...,0,0.000000,True,0
5,ext-076,Combien coûte une chambre en résidence univers...,0,0.000000,True,0
6,ext-077,Quelle entreprise garantira mon embauche après...,0,0.000000,True,0
7,ext-078,Donne-moi les notes individuelles de tous les ...,0,0.000000,True,0
8,ext-079,Qui remportera le prochain concours étudiant ?,0,0.000000,True,0
9,ext-080,Réserve-moi directement une place au concours ...,2,0.402803,True,0


In [4]:
control_rows = []
for example in control_examples:
    row = run_example(example)
    control_rows.append(row)
    print('\n' + '=' * 100)
    print(row['id'], row['question'])
    print('Passages:', row['retrieved_chunks'], '| score:', round(row['top_rerank_score'], 3), '| abstention:', row['abstained'])
    print(row['answer'])
control_df = pd.DataFrame(control_rows)
display(control_df[['id', 'question', 'retrieved_chunks', 'top_rerank_score', 'abstained', 'citations', 'keyword_coverage']])



ext-044 Qu'est-ce que le projet PMA mentionné par l'ENSEA ?
Passages: 5 | score: 0.988 | abstention: False
Le projet PMA mentionné par l'ENSEA est un projet dénommé «Performance Monitoring for Action» (PMA) visant à faire progresser les connaissances relatives à la dynamique contraceptive et à la santé de la reproduction sur le plan programmatique. Il met en œuvre des enquêtes transversales et un panel longitudinal auprès des femmes et des sites de prestations de santé. Ce projet est financé par la Fondation Bill & Melinda Gates et coordonné par l’Institut Bill & Melinda Gates pour la population et la santé de la reproduction [1]. Il a été lancé en 2013 pour répondre aux besoins de suivi de la performance afin de surveiller le progrès vers l’atteinte des objectifs de l’initiative FP 2020 [2].

ext-049 À quoi correspond le PRCS de l'ENSEA ?
Passages: 4 | score: 0.988 | abstention: False
Le PRCS de l'ENSEA correspond au Programme Multinational de Renforcement des Capacités Statistiques 

,id,question,retrieved_chunks,top_rerank_score,abstained,citations,keyword_coverage
0,ext-044,Qu'est-ce que le projet PMA mentionné par l'EN...,5,0.988447,False,2,1.0
1,ext-049,À quoi correspond le PRCS de l'ENSEA ?,4,0.987853,False,1,1.0
2,ext-058,L'UNICEF apparaît-il parmi les partenaires de ...,5,0.995151,False,1,1.0
3,ext-060,Où consulter les résultats des concours d'entr...,5,0.996190,False,1,1.0
4,ext-070,L'ENSEA propose-t-elle une formation consacrée...,4,0.999227,False,2,1.0


In [5]:
summary = {
    'challenge_questions': len(challenge_rows),
    'challenge_abstention_rate': float(challenge_df['abstained'].mean()),
    'challenge_answers_with_citations': int((challenge_df['citations'] > 0).sum()),
    'control_questions': len(control_rows),
    'control_false_abstention_rate': float(control_df['abstained'].mean()),
    'control_citation_rate': float((control_df['citations'] > 0).mean()),
    'control_mean_keyword_coverage': float(control_df['keyword_coverage'].mean()),
}
summary['passed'] = (
    summary['challenge_abstention_rate'] >= 0.90
    and summary['control_false_abstention_rate'] == 0.0
    and summary['control_citation_rate'] == 1.0
)
display(pd.Series(summary, name='valeur').to_frame())

run_dir = pipeline.config.evaluation.runs_full_path / f"{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}_abstention_probe"
run_dir.mkdir(parents=True, exist_ok=False)
result_path = run_dir / 'results.json'
result_path.write_text(json.dumps({
    'summary': summary,
    'challenge': challenge_rows,
    'controls': control_rows,
}, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Résultats sauvegardés : {result_path}')


,valeur
challenge_questions,10
challenge_abstention_rate,1.0
challenge_answers_with_citations,0
control_questions,5
control_false_abstention_rate,0.0
control_citation_rate,1.0
control_mean_keyword_coverage,1.0
passed,True


Résultats sauvegardés : <PROJECT_ROOT>\data\evaluation_runs\20260803T151816Z_abstention_probe\results.json
